# Backtest explorer

Compare backtest runs and inspect the data behind them. Everything reads local files:

- `data/backtests.db`: every run from `python scripts/run_backtest.py --notes "..."`
- `data/chess_tournament_data.db`: the Railway copy from `python scripts/pull_snapshot.py`

**Reading the scores.** Log loss and Brier are lower-is-better. They only compare fairly
across runs covering the same events. `p_top` is the tradable probability
(`p_participate` x `P_topN_given_play`).

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src import backtest_runs as br
from src.config import DB_PATH

pd.set_option('display.width', 160, 'display.max_columns', 40, 'display.max_rows', 200)

# Each model keeps one color and marker in every chart. The marker shape means
# identity never depends on color alone.
MODEL_STYLES = {
    'score':               {'color': '#2a78d6', 'marker': 'o'},
    'perf':                {'color': '#eb6834', 'marker': 's'},
    'score_smooth':        {'color': '#1baf7a', 'marker': '^'},
    'score_smooth_narrow': {'color': '#7a2f8f', 'marker': 'D'},
    'score_smooth_3':      {'color': '#1baf7a', 'marker': '^'},
    'score_smooth_10':     {'color': '#eb6834', 'marker': 's'},
}
OTHER_STYLE = {'color': '#8a8983', 'marker': 'x'}
MUTED, GRID = '#52514e', '#e4e3df'

plt.rcParams.update({
    'figure.dpi': 110, 'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': GRID,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.labelcolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
    'lines.linewidth': 2, 'lines.markersize': 6, 'font.size': 10,
})


def style(model):
    return MODEL_STYLES.get(model, OTHER_STYLE)


def share_legend(fig, axes):
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), bbox_to_anchor=(0.5, 1.07))

## 1. Runs

Every recorded run, newest first.

In [ ]:
runs = br.list_runs()
display(runs[['market', 'status', 'started_at', 'git_commit', 'git_dirty', 'models', 'n_events', 'n_sims',
              'start_date', 'attendance_start', 'notes', 'error']])

## 2. Choose runs

`RUN_IDS` drives every section below; it defaults to the latest completed run. Charts that
compare models use the first run listed. `run_settings` shows exactly what that run used.

In [ ]:
RUN_IDS = runs.index[runs['status'] == 'completed'][:1].tolist()   # e.g. [4, 3]
assert RUN_IDS, 'No completed runs yet. Run: python scripts/run_backtest.py --notes "baseline"'

predictions = br.load_predictions(RUN_IDS)
scored = br.score_predictions(predictions)
summary = br.summarize(scored)
PRIMARY = RUN_IDS[0]
primary = scored[scored['run_id'] == PRIMARY]
models = [m for m in MODEL_STYLES if m in set(primary['model'])] + sorted(set(primary['model']) - set(MODEL_STYLES))
NS = sorted(primary['n'].unique())
br.run_settings(PRIMARY)

## 3. Accuracy by model

`predicted_per_event` should sit close to `actual_per_event`. `hit_capture` below 1.0 means
some real finishers were outside the prediction pool.

In [ ]:
display(summary.round(4))

panels = [n for n in NS if n in (1, 3, 5, 8)]
fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 0.42 * len(models) + 1.4), sharey=True)
axes = np.atleast_1d(axes)
for ax, n in zip(axes, panels):
    values = summary.loc[PRIMARY].xs(n, level='n')['log_loss'].reindex(models)
    ax.barh(range(len(models)), values, height=0.55, color=[style(m)['color'] for m in models])
    for y, v in enumerate(values):
        ax.annotate(f'{v:.4f}', (v, y), xytext=(4, 0), textcoords='offset points', va='center', fontsize=8.5, color=MUTED)
    ax.set_title('Winner' if n == 1 else f'Top {n}')
    ax.set_xlim(0, values.max() * 1.28)
    ax.grid(axis='y', visible=False)
    ax.set_xlabel('Log loss (lower is better)')
axes[0].set_yticks(range(len(models)), models)
axes[0].invert_yaxis()
fig.suptitle(f'Run {PRIMARY}: log loss by model', x=0.01, ha='left', y=1.02, fontweight='bold')
fig.tight_layout()
plt.show()

## 4. Calibration

Points on the dashed line mean a predicted X% happened X% of the time. Above the line the
model was too low; below it, too high. Axes use a square-root scale to spread out small
probabilities. The chart skips bins with fewer than `MIN_BIN_COUNT` observations because they are
mostly noise; the table below it shows every bin.

In [ ]:
MIN_BIN_COUNT = 20

calib = br.calibration(primary)
panels = [n for n in NS if n in (1, 3, 5, 8)]
fig, axes = plt.subplots(1, len(panels), figsize=(3.3 * len(panels), 3.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, n in zip(axes, panels):
    ax.plot([0, 1], [0, 1], ls='--', lw=1, color='#8a8983')
    for model in models:
        g = calib[(calib['model'] == model) & (calib['n'] == n) & (calib['count'] >= MIN_BIN_COUNT)]
        ax.plot(g['predicted'], g['observed'], label=model, **style(model))
    ax.set_xscale('function', functions=(np.sqrt, np.square))
    ax.set_yscale('function', functions=(np.sqrt, np.square))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([0.01, 0.1, 0.25, 0.5, 1], ['1%', '10%', '25%', '50%', '100%'])
    ax.set_yticks([0.01, 0.1, 0.25, 0.5, 1], ['1%', '10%', '25%', '50%', '100%'])
    ax.set_title('Winner' if n == 1 else f'Top {n}')
    ax.set_xlabel('Predicted')
axes[0].set_ylabel('Observed')
share_legend(fig, axes)
fig.tight_layout()
plt.show()

display(calib.round(4))

## 5. Attendance

Attendance is the same for every model that shares a player pool. The chart shows whether the
model expected the right number of players from its pool at each event.

In [ ]:
display(br.attendance_summary(scored).round(4))

model = models[0]
one = primary[(primary['n'] == NS[0]) & (primary['model'] == model)]
by_event = one.groupby('tourn_date').agg(predicted=('p_participate', 'sum'), actual=('played', 'sum'))
dates = pd.to_datetime(by_event.index)

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(dates, by_event['predicted'], color='#2a78d6', marker='o', label='Predicted')
ax.plot(dates, by_event['actual'], color='#eb6834', marker='s', label='Actual')
ax.set_ylim(bottom=0)
ax.set_ylabel('Pool players attending')
ax.set_title(f'Run {PRIMARY}: expected vs actual attendance ({model} pool)', loc='left')
ax.legend(loc='lower left', ncol=2)
fig.autofmt_xdate()
plt.show()

attendance_calib = br.calibration(primary[primary['n'] == NS[0]], probability='p_participate', outcome='played')
display(attendance_calib[attendance_calib['model'] == model].round(4))

## 6. Model difference over time

Per-event log loss relative to `BASELINE`. Below zero, that model beat the baseline that week.

In [ ]:
BASELINE, CHART_N = 'score', 1

per_event = (primary[primary['n'] == CHART_N].assign(
    loss=lambda d: -(d['hit'] * np.log(d['p_top'].clip(1e-6, 1 - 1e-6))
                     + (1 - d['hit']) * np.log(1 - d['p_top'].clip(1e-6, 1 - 1e-6))))
    .pivot_table(index='tourn_date', columns='model', values='loss', aggfunc='mean'))

if BASELINE in per_event and per_event.shape[1] > 1:
    delta = per_event.drop(columns=BASELINE).sub(per_event[BASELINE], axis=0)
    dates = pd.to_datetime(delta.index)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.axhline(0, color=MUTED, lw=1)
    for model in [m for m in models if m in delta]:
        ax.plot(dates, delta[model], label=model, **style(model))
    ax.set_ylabel(f'Log loss minus {BASELINE}')
    ax.set_title(f'Run {PRIMARY}: {"winner" if CHART_N == 1 else f"top {CHART_N}"} log loss vs {BASELINE}', loc='left')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=len(delta.columns))
    fig.autofmt_xdate()
    plt.show()
    display(delta.round(5))
else:
    print('This run has only one model, so there is nothing to compare.')

## 7. Single-event drill-down

The highest-probability players at one event, and what actually happened.

In [ ]:
EVENT, EVENT_MODEL, EVENT_N = primary['tourn_date'].max(), models[0], 1

view = (primary[(primary['tourn_date'] == EVENT) & (primary['model'] == EVENT_MODEL) & (primary['n'] == EVENT_N)]
        .merge(predictions.loc[predictions['run_id'] == PRIMARY, ['tourn_date', 'model', 'username', 'actual_rank']],
               on=['tourn_date', 'model', 'username'])
        .sort_values('p_top', ascending=False)
        [['username', 'p_participate', 'p_given_play', 'p_top', 'played', 'actual_rank', 'hit']])
print(f'{EVENT} | {EVENT_MODEL} | top {EVENT_N}')
display(view.head(15).round(4))

## 8. Compare two runs

Set `COMPARE = (older, newer)` to see how settings changed the scores. Negative `log_loss_change`
means the newer run improved. Only events and models in both runs are compared.

In [ ]:
COMPARE = None   # e.g. (3, 4)

if COMPARE is None:
    print('Set COMPARE to a pair of run_ids from the table in section 1.')
else:
    a, b = br.load_predictions([COMPARE[0]]), br.load_predictions([COMPARE[1]])
    shared = a[['tourn_date', 'model']].drop_duplicates().merge(b[['tourn_date', 'model']].drop_duplicates())
    a_summary = br.summarize(br.score_predictions(a.merge(shared))).droplevel('run_id')
    b_summary = br.summarize(br.score_predictions(b.merge(shared))).droplevel('run_id')
    changes = pd.DataFrame({
        'events': a_summary['events'],
        'log_loss_older': a_summary['log_loss'], 'log_loss_newer': b_summary['log_loss'],
        'log_loss_change': b_summary['log_loss'] - a_summary['log_loss'],
    })
    display(changes.round(5))
    a_settings, b_settings = br.run_settings(COMPARE[0]), br.run_settings(COMPARE[1])
    display({k: (a_settings.get(k), b_settings.get(k)) for k in sorted(a_settings | b_settings)
             if a_settings.get(k) != b_settings.get(k)})

## 9. Data overview

A starting point for EDA on the Railway snapshot.

In [ ]:
snapshot = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", snapshot)['name']
display(pd.Series({t: snapshot.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0] for t in tables},
                  name='rows').to_frame())

standings = pd.read_sql_query('SELECT date, username, rank, score FROM titled_tuesday_standings', snapshot)
standings['date'] = pd.to_datetime(standings['date'], format='mixed', utc=True).dt.tz_localize(None)
field = standings.groupby('date')['username'].nunique()

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(field.index, field.values, color='#2a78d6')
ax.set_ylim(bottom=0)
ax.set_ylabel('Players')
ax.set_title('Field size by event', loc='left')
plt.show()

recent = standings[standings['date'] >= standings['date'].max() - pd.Timedelta(weeks=26)]
display(recent.groupby('username').agg(events=('date', 'nunique'), avg_score=('score', 'mean'), best_rank=('rank', 'min'))
        .sort_values(['events', 'avg_score'], ascending=False).head(20).round(2))